In [3]:
"""Composite Mishler spectral-motif pipeline.

Edit MAT_FILE below so it points to a MATLAB .mat file containing:

    LFP : MATLAB cell array arranged as animals x sessions
          (one continuous LFP trace per non-empty cell)
    fs  : scalar LFP sampling rate in Hz

Pipeline
--------
1. Load LFP and fs.
2. Flatten LFP exactly like MATLAB LFP(:), using column-major order.
3. Apply a zero-phase 59-61 Hz bandstop to each raw LFP trace.
4. Run the SmoothEnv port:
       [1,5], [2,6], ..., [76,80] Hz bands
       Hilbert amplitude
       5-s Gaussian smoothing
       first/last 10 s set to NaN
       within-band z-scoring
5. AFTER SmoothEnv is finished, identify noisy times from the bandstopped LFP
   using the filterLFPNoise port and set those same time indices in the
   SmoothEnv output to NaN.
6. Remove every time point containing any NaN.
7. Store the cleaned SmoothEnv sessions as an N x 1 MATLAB-style cell vector
   of bands x time arrays.
8. Run the getMotifs port on those cleaned sessions.
9. Save smoothLFPNorm, finalMotifs, fs, and motif diagnostics to a new .mat file.

Dependencies
------------
numpy
scipy
"""

from dataclasses import dataclass, field
from pathlib import Path
import time
import warnings

import numpy as np
from scipy import signal
from scipy.cluster.hierarchy import linkage
from scipy.io import loadmat, savemat
from scipy.spatial.distance import pdist


# -------------------------------------------------------------------------
# EDIT THESE PATHS
# -------------------------------------------------------------------------
MAT_FILE = Path(r"C:\Paper Backup\LFP Data.mat")

# Directory where the motif output file will be saved.
SAVE_DIR = Path(r"C:\Paper Backup\motifs")

# Optional custom output filename. Leave as None to use:
# <input_filename>_motifs.mat
OUTPUT_FILE = None

# Matches the MATLAB filterLFPNoise usage.
STDEV_THRESH = 10.0

# getMotifs settings.
N_NULL = 20
RNG_SEED = None
VERBOSE = True


DEFAULT_FREQ_BANDS = np.column_stack((
    np.arange(1.0, 77.0),
    np.arange(5.0, 81.0),
))


def mround(x: float) -> int:
    """MATLAB round(): halves go away from zero."""
    return int(np.sign(x) * np.floor(abs(x) + 0.5))


# -------------------------------------------------------------------------
# Filtering / SmoothEnv
# -------------------------------------------------------------------------

def butterfilter(fs: float, low: float, high: float, order: int = 6) -> np.ndarray:
    """Butterworth bandpass as second-order sections, matching butterfilter.m."""
    return signal.butter(
        order // 2,
        [low, high],
        btype="bandpass",
        fs=fs,
        output="sos",
    )


def bandstop_60hz(x: np.ndarray, fs: float) -> np.ndarray:
    """Zero-phase 59-61 Hz bandstop performed immediately after LFP import."""
    if fs / 2 <= 61:
        raise ValueError(
            "Sampling rate must be greater than 122 Hz for a 59-61 Hz bandstop."
        )

    sos = signal.butter(
        3,
        [59.0, 61.0],
        btype="bandstop",
        fs=fs,
        output="sos",
    )
    return signal.sosfiltfilt(sos, np.asarray(x, dtype=float))


def gausswin(n: int, alpha: float = 2.5) -> np.ndarray:
    """MATLAB gausswin(n, alpha): std = (n - 1) / (2 * alpha)."""
    return signal.windows.gaussian(n, std=(n - 1) / (2 * alpha))


def fir_filtfilt(b: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Equivalent to scipy.signal.filtfilt(b, 1, x, axis=0), using FFT convolution."""
    b = np.asarray(b, dtype=float)
    x = np.asarray(x, dtype=float)

    ntaps = b.size
    padlen = 3 * ntaps

    if x.shape[0] <= padlen:
        raise ValueError(
            f"signal needs more than {padlen} samples for this smoothing kernel"
        )

    b_nd = b.reshape((-1,) + (1,) * (x.ndim - 1))

    def one_pass(y):
        held = np.concatenate(
            [np.repeat(y[:1], ntaps - 1, axis=0), y],
            axis=0,
        )
        return signal.fftconvolve(held, b_nd, mode="valid", axes=0)

    ext = np.concatenate(
        [
            2 * x[:1] - x[padlen:0:-1],
            x,
            2 * x[-1:] - x[-2:-padlen - 2:-1],
        ],
        axis=0,
    )

    y = one_pass(ext)
    y = one_pass(y[::-1])[::-1]
    return y[padlen:-padlen]


def smooth_env(
    lfp: np.ndarray,
    fs: float,
    freq_bands: np.ndarray = DEFAULT_FREQ_BANDS,
    smooth_sec: float = 5.0,
    edge_sec: float = 10.0,
) -> np.ndarray:
    """SmoothEnv port.

    Input is expected to have already received the 59-61 Hz bandstop.
    Returns time x bands.
    """
    x = np.asarray(lfp, dtype=float).ravel()
    freq_bands = np.atleast_2d(np.asarray(freq_bands, dtype=float))

    filt = np.empty((x.size, len(freq_bands)))
    for f, (low, high) in enumerate(freq_bands):
        filt[:, f] = signal.sosfiltfilt(
            butterfilter(fs, low, high),
            x,
        )

    env = np.abs(signal.hilbert(filt, axis=0))

    w = gausswin(mround(smooth_sec * fs), 1)
    w /= w.sum()
    smooth = fir_filtfilt(w, env)

    edge = mround(edge_sec * fs)
    smooth[:edge, :] = np.nan
    smooth[-edge:, :] = np.nan

    mu = np.nanmean(smooth, axis=0)
    sigma = np.nanstd(smooth, axis=0, ddof=1)
    return (smooth - mu) / sigma


# -------------------------------------------------------------------------
# filterLFPNoise
# -------------------------------------------------------------------------

def filter_lfp_noise(
    fs: float,
    orig_lfp: np.ndarray,
    stdev_thresh: float,
) -> np.ndarray:
    """Port of filterLFPNoise.m.

    Detects |LFP| > mean(|LFP|) + stdev_thresh * SD(|LFP|), expands each
    detected point by +/- 3 s, and also includes the first and last 10 s.

    Returned indices are zero-based Python indices.
    """
    x = np.asarray(orig_lfp, dtype=float).ravel()

    noise_samp = mround(3.0 * fs)
    snip_time = np.arange(-noise_samp, noise_samp + 1)

    mag = np.abs(x)
    threshold = np.nanmean(mag) + stdev_thresh * np.nanstd(mag, ddof=1)
    noise_idx = np.flatnonzero(mag > threshold)

    if noise_idx.size:
        artifact_idx = (noise_idx[:, None] + snip_time[None, :]).ravel()
        artifact_idx = artifact_idx[
            (artifact_idx >= 0) & (artifact_idx < x.size)
        ]
        artifact_idx = np.unique(artifact_idx)
    else:
        artifact_idx = np.array([], dtype=int)

    edge_samp = mround(10.0 * fs)
    start_idx = np.arange(0, min(edge_samp, x.size))
    end_idx = np.arange(max(0, x.size - edge_samp), x.size)

    return np.unique(
        np.concatenate((start_idx, artifact_idx, end_idx))
    )


# -------------------------------------------------------------------------
# MAT-file preprocessing
# -------------------------------------------------------------------------

def _scalar_fs(value) -> float:
    arr = np.asarray(value, dtype=float).squeeze()
    if arr.size != 1:
        raise ValueError("'fs' must be a scalar sampling rate.")
    fs = float(arr)
    if not np.isfinite(fs) or fs <= 0:
        raise ValueError("'fs' must be positive and finite.")
    return fs


def _flatten_lfp_cells(lfp) -> np.ndarray:
    """Flatten the cell matrix exactly like MATLAB LFP(:)."""
    arr = np.asarray(lfp)

    if arr.dtype != object:
        raise ValueError(
            "'LFP' must be a MATLAB cell array with one continuous trace per cell."
        )

    return arr.reshape(-1, order="F")


def preprocess_lfp_sessions(lfp, fs: float, stdev_thresh: float):
    """Return a list of cleaned bands x time SmoothEnv arrays."""
    sessions = _flatten_lfp_cells(lfp)
    cleaned_sessions = []

    for i, cell in enumerate(sessions, start=1):
        raw = np.asarray(cell, dtype=float).squeeze()

        if raw.size == 0:
            raise ValueError(f"LFP session {i} is empty.")
        if raw.ndim != 1:
            raise ValueError(
                f"LFP session {i} must be one continuous 1-D trace; got {raw.shape}."
            )
        if not np.isfinite(raw).all():
            raise ValueError(
                f"LFP session {i} contains NaN or Inf before preprocessing."
            )

        # Bandstop happens immediately after the session is imported.
        lfp_bandstopped = bandstop_60hz(raw, fs)

        # KEY ORDER: SmoothEnv is completed before noise detection.
        smooth = smooth_env(lfp_bandstopped, fs)

        # Only now identify noisy times.
        noise_idx = filter_lfp_noise(
            fs=fs,
            orig_lfp=lfp_bandstopped,
            stdev_thresh=stdev_thresh,
        )

        # Add noise NaNs to the NaNs already produced by SmoothEnv.
        smooth[noise_idx, :] = np.nan

        # Convert time x bands -> bands x time.
        smooth = smooth.T

        # Remove every time column containing a NaN.
        keep = np.isfinite(smooth).all(axis=0)
        clean = smooth[:, keep]

        if clean.shape[1] == 0:
            raise ValueError(f"LFP session {i} has no samples remaining after cleaning.")

        cleaned_sessions.append(clean)

        print(
            f"Preprocessed session {i}/{len(sessions)}: "
            f"{raw.size} raw -> {clean.shape[1]} clean samples"
        )

    return cleaned_sessions


CORR_THRESHOLDS = (0.4, 0.5, 0.6, 0.7, 0.8)


@dataclass
class MotifResult:
    motifs: np.ndarray
    """(bands, 2 * n_motifs). Column i + n_motifs equals -column i."""
    thresholds: tuple
    null_max_size: np.ndarray
    """Per threshold: size of the largest motif in any shifted null dataset."""
    candidate_sizes: list
    """Per threshold: sizes of the motif candidates found in the real data."""
    n_kept: np.ndarray
    """Per threshold: candidates that beat the null and appear in every session."""
    n_timepoints: int
    """Non-trivial ~1 Hz timepoints in the real data."""
    shifts: np.ndarray = field(repr=False)
    """(bands, 1 + n_null) circular shifts in samples; column 0 is the real data."""

    @property
    def n_motifs(self) -> int:
        return self.motifs.shape[1] // 2


@dataclass
class _Tree:
    a: np.ndarray       # left child of internal node n + k
    b: np.ndarray       # right child of internal node n + k
    parent: np.ndarray  # parent node id, -1 for the root
    size: np.ndarray    # timepoints under each node
    rep: np.ndarray     # smallest leaf position under each node (MATLAB list order)
    r: np.ndarray       # correlation between the two children's expression time courses
    sums: np.ndarray    # sum of member sign patterns under each node


def _expression_corr(t1, t2, second_moment, mean):
    """Row-wise corr(allCorr' * t1, allCorr' * t2) over all timepoints.

    Pearson correlation of projections only needs the data's mean and second
    moment, so this costs O(bands^2) per pair instead of O(timepoints).
    """
    m1, m2 = t1 @ mean, t2 @ mean
    c1, c2 = t1 @ second_moment, t2 @ second_moment
    e11, e22 = (c1 * t1).sum(1), (c2 * t2).sum(1)
    cov = (c1 * t2).sum(1) - m1 * m2
    v1, v2 = e11 - m1 ** 2, e22 - m2 ** 2
    flat = (v1 <= 1e-12 * np.maximum(e11, 1e-300)) | (v2 <= 1e-12 * np.maximum(e22, 1e-300))
    with np.errstate(invalid="ignore", divide="ignore"):
        r = cov / np.sqrt(v1 * v2)
    r[flat] = np.nan  # MATLAB corr() of a constant vector is NaN, which never passes r >= thresh
    return r


def _build_tree(Z, leaf_sums, leaf_sizes, second_moment, mean) -> _Tree:
    n = leaf_sums.shape[0]
    a = Z[:, 0].astype(np.int64)
    b = Z[:, 1].astype(np.int64)
    parent = np.full(2 * n - 1, -1, dtype=np.int64)
    parent[a] = np.arange(n, 2 * n - 1)
    parent[b] = np.arange(n, 2 * n - 1)

    sums = np.empty((2 * n - 1, leaf_sums.shape[1]))
    sums[:n] = leaf_sums
    size = np.empty(2 * n - 1)
    size[:n] = leaf_sizes
    rep = np.empty(2 * n - 1, dtype=np.int64)
    rep[:n] = np.arange(n)
    for k in range(n - 1):
        sums[n + k] = sums[a[k]] + sums[b[k]]
        size[n + k] = size[a[k]] + size[b[k]]
        rep[n + k] = min(rep[a[k]], rep[b[k]])

    templates = sums / size[:, None]
    r = _expression_corr(templates[a], templates[b], second_moment, mean)
    return _Tree(a=a, b=b, parent=parent, size=size, rep=rep, r=r, sums=sums)


def _consolidate(tree: _Tree, thresh: float) -> np.ndarray:
    """Tree nodes that end the MATLAB inner consolidation loop as motif candidates.

    Returned in MATLAB's order: branches still live at the end (by list position),
    then branches frozen for failing the threshold (by the sweep they froze in,
    then list position).
    """
    n = tree.a.size + 1
    live = np.zeros(2 * n - 1, dtype=bool)
    live[:n] = True
    ready = np.zeros(2 * n - 1, dtype=np.int64)  # sweep at whose start a node is live
    ready[:n] = 1
    frozen_in = np.zeros(2 * n - 1, dtype=np.int64)
    merge = (tree.r >= thresh).tolist()
    a, b = tree.a.tolist(), tree.b.tolist()
    live_l, ready_l, frozen_l = live.tolist(), ready.tolist(), frozen_in.tolist()
    for k in range(n - 1):
        ak, bk = a[k], b[k]
        if live_l[ak] and live_l[bk]:
            sweep = max(ready_l[ak], ready_l[bk])
            if merge[k]:
                live_l[n + k] = True
                ready_l[n + k] = sweep + 1
            else:
                frozen_l[ak] = frozen_l[bk] = sweep
    live = np.array(live_l)
    frozen_in = np.array(frozen_l)

    parent_live = np.zeros_like(live)
    has_parent = tree.parent >= 0
    parent_live[has_parent] = live[tree.parent[has_parent]]
    still_live = np.flatnonzero(live & ~parent_live & (frozen_in == 0))
    frozen = np.flatnonzero(frozen_in > 0)
    still_live = still_live[np.argsort(tree.rep[still_live], kind="stable")]
    frozen = frozen[np.lexsort((tree.rep[frozen], frozen_in[frozen]))]
    return np.concatenate([still_live, frozen])


def _leaf_owner(tree: _Tree, nodes: np.ndarray) -> np.ndarray:
    """For each leaf, the index (into `nodes`) of the candidate subtree containing it."""
    owner = np.full(tree.parent.size, -1, dtype=np.int64)
    owner[nodes] = np.arange(nodes.size)
    parent = tree.parent.tolist()
    owner_l = owner.tolist()
    for v in range(len(owner_l) - 1, -1, -1):
        if owner_l[v] < 0:
            owner_l[v] = owner_l[parent[v]]
    n = tree.a.size + 1
    return np.array(owner_l[:n], dtype=np.int64)


def _correlation_linkage(points: np.ndarray) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        d = pdist(points, "correlation")
    return linkage(np.nan_to_num(d, nan=1.0), "average")


def _candidates(X, tree0, thresh, second_moment, mean, need_labels):
    """One threshold of getMotifs' consolidation, outer re-clustering loop included.

    Returns (sizes, labels): candidate sizes in MATLAB order, and for each
    timepoint the index of its candidate (None unless need_labels).
    """
    nodes = _consolidate(tree0, thresh)
    labels = _leaf_owner(tree0, nodes) if need_labels else None
    sums, sizes = tree0.sums[nodes], tree0.size[nodes]
    history = [nodes.size]
    while not (len(history) > 1 and history[-1] == history[-2]) and sizes.size > 1:
        tree = _build_tree(_correlation_linkage(sums / sizes[:, None]), sums, sizes, second_moment, mean)
        nodes = _consolidate(tree, thresh)
        if need_labels:
            labels = _leaf_owner(tree, nodes)[labels]
        sums, sizes = tree.sums[nodes], tree.size[nodes]
        history.append(nodes.size)
    return sizes, labels


def candidate_partition(X: np.ndarray, thresh: float):
    """Motif candidates for (timepoints, bands) sign data at one threshold, before any null test."""
    X = np.asarray(X, dtype=float)
    second_moment, mean = X.T @ X / X.shape[0], X.mean(axis=0)
    tree0 = _build_tree(_correlation_linkage(X), X, np.ones(X.shape[0]), second_moment, mean)
    return _candidates(X, tree0, thresh, second_moment, mean, need_labels=True)


def _pca_uncentered(F):
    """MATLAB pca(F, 'Centered', 'off'), including its coefficient sign convention."""
    _, _, vt = np.linalg.svd(F, full_matrices=False)
    coeff = vt.T
    biggest = np.argmax(np.abs(coeff), axis=0)
    sign = np.sign(coeff[biggest, np.arange(coeff.shape[1])])
    sign[sign == 0] = 1
    coeff = coeff * sign
    return coeff, F @ coeff


def _null_shifts(n_bands, fs, n_null, rng):
    shift_times = np.array([mround(fs * 20 * k) for k in range(n_bands)])
    perms = np.stack([shift_times[rng.permutation(n_bands)] for _ in range(100)])
    perms = np.unique(perms, axis=0)  # sorted rows, as MATLAB's unique(..., 'rows')
    return np.column_stack([np.zeros(n_bands, dtype=np.int64), perms[:n_null].T])


def _binarize(sessions, shift, hop):
    shifted = [np.stack([np.roll(s[f], shift[f]) for f in range(s.shape[0])]) for s in sessions]
    signs = np.sign(np.concatenate(shifted, axis=1)[:, ::hop])
    nontrivial = np.abs(signs.sum(axis=0)) != signs.shape[0]
    return signs[:, nontrivial].T, nontrivial


def get_motifs(
    lfp_in,
    fs: float,
    corr_thresholds=CORR_THRESHOLDS,
    n_null: int = 20,
    seed=None,
    verbose: bool = False,
) -> MotifResult:
    """Port of getMotifs(LFPIn, LFPfs).

    lfp_in: list of (bands, time) arrays, one per session, same bands in the same
    order. Must not contain NaN. fs: sampling rate of the time axis in Hz.
    """
    sessions = [np.asarray(s, dtype=float) for s in lfp_in]
    n_bands = sessions[0].shape[0]
    if any(s.ndim != 2 or s.shape[0] != n_bands for s in sessions):
        raise ValueError("every session must be a (bands, time) array with the same number of bands")
    if not all(np.isfinite(s).all() for s in sessions):
        raise ValueError("input contains NaN/inf; trim SmoothEnv's NaN edges before calling get_motifs")

    hop = mround(fs)
    session_of = np.concatenate([np.full(s.shape[1], i) for i, s in enumerate(sessions)])[::hop]
    shifts = _null_shifts(n_bands, fs, n_null, np.random.default_rng(seed))
    thresholds = tuple(corr_thresholds)

    null_max = np.zeros(len(thresholds))
    real = None
    for s in range(shifts.shape[1]):
        t0 = time.time()
        X, nontrivial = _binarize(sessions, shifts[:, s], hop)
        second_moment, mean = X.T @ X / X.shape[0], X.mean(axis=0)
        tree0 = _build_tree(_correlation_linkage(X), X, np.ones(X.shape[0]), second_moment, mean)
        per_thresh = [_candidates(X, tree0, t, second_moment, mean, need_labels=(s == 0)) for t in thresholds]
        if s == 0:
            real = (X, session_of[nontrivial], per_thresh)
        else:
            null_max = np.maximum(null_max, [sizes.max() for sizes, _ in per_thresh])
        if verbose:
            kind = "real data" if s == 0 else f"null {s}/{shifts.shape[1] - 1}"
            print(f"  [{kind}] {X.shape[0]} timepoints, candidates per threshold: "
                  f"{[sizes.size for sizes, _ in per_thresh]} ({time.time() - t0:.1f}s)", flush=True)

    X, session_ids, per_thresh = real
    templates, n_kept = [], []
    for (sizes, labels), limit in zip(per_thresh, null_max):
        kept = 0
        for k in np.flatnonzero(sizes > limit):
            members = labels == k
            if np.unique(session_ids[members]).size == len(sessions):
                templates.append(X[members].mean(axis=0))
                kept += 1
        n_kept.append(kept)

    result = MotifResult(
        motifs=np.zeros((n_bands, 0)),
        thresholds=thresholds,
        null_max_size=null_max,
        candidate_sizes=[sizes for sizes, _ in per_thresh],
        n_kept=np.array(n_kept),
        n_timepoints=X.shape[0],
        shifts=shifts,
    )
    if not templates:
        return result

    coeff, score = _pca_uncentered(np.array(templates))
    magnitude = np.abs(score)
    _, best_axis = np.nonzero(magnitude == magnitude.max(axis=1, keepdims=True))
    axes = np.flatnonzero(np.bincount(best_axis, minlength=coeff.shape[1]) > 1)
    if axes.size == 0:
        return result

    proj = X @ coeff[:, axes]
    proj = np.hstack([proj, -proj])
    best = proj.max(axis=1)
    motifs = np.full((n_bands, axes.size), np.nan)
    for i in range(axes.size):
        pooled = np.vstack([X[proj[:, i] == best], -X[proj[:, i + axes.size] == best]])
        if pooled.size:
            motifs[:, i] = pooled.mean(axis=0)
    result.motifs = np.hstack([motifs, -motifs])
    return result


# -------------------------------------------------------------------------
# Run the full pipeline
# -------------------------------------------------------------------------

def main():
    if not MAT_FILE.exists():
        raise FileNotFoundError(
            f"MAT_FILE does not exist:\n{MAT_FILE}\n\n"
            "Edit MAT_FILE near the top of this script."
        )

    try:
        mat = loadmat(MAT_FILE, squeeze_me=False, struct_as_record=False)
    except NotImplementedError as exc:
        raise RuntimeError(
            "scipy.io.loadmat cannot read MATLAB v7.3 HDF5 MAT files. "
            "Save the input as a pre-v7.3 MAT file before running this script."
        ) from exc

    if "LFP" not in mat:
        raise KeyError("The MAT file does not contain a variable named 'LFP'.")
    if "LFPfs" not in mat:
        raise KeyError("The MAT file does not contain a variable named 'fs'.")

    fs = _scalar_fs(mat["LFPfs"])

    print(f"Loaded: {MAT_FILE}")
    print(f"Sampling rate: {fs:g} Hz")

    smooth_lfp_norm = preprocess_lfp_sessions(
        lfp=mat["LFP"],
        fs=fs,
        stdev_thresh=STDEV_THRESH,
    )

    print("\nRunning getMotifs...")
    motif_result = get_motifs(
        lfp_in=smooth_lfp_norm,
        fs=fs,
        n_null=N_NULL,
        seed=RNG_SEED,
        verbose=VERBOSE,
    )

    final_motifs = motif_result.motifs

    # MATLAB-style N x 1 cell vector of cleaned sessions.
    smooth_cell = np.empty((len(smooth_lfp_norm), 1), dtype=object)
    for i, session in enumerate(smooth_lfp_norm):
        smooth_cell[i, 0] = session

    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    output_file = (
        SAVE_DIR / Path(OUTPUT_FILE).name
        if OUTPUT_FILE is not None
        else SAVE_DIR / f"{MAT_FILE.stem}_motifs.mat"
    )

    # Variable-length candidate-size arrays are saved as a MATLAB cell array.
    candidate_sizes = np.empty((len(motif_result.candidate_sizes), 1), dtype=object)
    for i, sizes in enumerate(motif_result.candidate_sizes):
        candidate_sizes[i, 0] = np.asarray(sizes)

    savemat(
        output_file,
        {
            "smoothLFPNorm": smooth_cell,
            "finalMotifs": final_motifs,
            "fs": np.array([[fs]], dtype=float),
            "corrThresholds": np.asarray(motif_result.thresholds, dtype=float),
            "nullMaxSize": motif_result.null_max_size,
            "candidateSizes": candidate_sizes,
            "nKept": motif_result.n_kept,
            "nTimepoints": np.array([[motif_result.n_timepoints]], dtype=float),
            "shifts": motif_result.shifts,
        },
        do_compression=True,
    )

    print(f"\nFinal motifs: {motif_result.n_motifs}")
    print(f"Saved: {output_file}")


if __name__ == "__main__":
    main()


Loaded: C:\Paper Backup\LFP Data.mat
Sampling rate: 1017.25 Hz
Preprocessed session 1/18: 2442112 raw -> 2349244 clean samples
Preprocessed session 2/18: 2443520 raw -> 2320198 clean samples
Preprocessed session 3/18: 2043392 raw -> 1992521 clean samples
Preprocessed session 4/18: 2498176 raw -> 2447305 clean samples
Preprocessed session 5/18: 2494720 raw -> 2468268 clean samples
Preprocessed session 6/18: 2714496 raw -> 2656491 clean samples
Preprocessed session 7/18: 2442752 raw -> 2357305 clean samples
Preprocessed session 8/18: 2442240 raw -> 2335807 clean samples
Preprocessed session 9/18: 2086912 raw -> 2032538 clean samples
Preprocessed session 10/18: 2452864 raw -> 2309496 clean samples
Preprocessed session 11/18: 3565184 raw -> 3502093 clean samples
Preprocessed session 12/18: 2547456 raw -> 2453386 clean samples
Preprocessed session 13/18: 2472448 raw -> 2414170 clean samples
Preprocessed session 14/18: 2454144 raw -> 2321232 clean samples
Preprocessed session 15/18: 1935488 

MatWriteError: Matrix too large to save with Matlab 5 format

Recovered finalMotifs shape: (76, 6)
Saved: C:\Paper Backup\motifs\finalMotifs.mat
